In [ ]:
import sys
from pathlib import Path
import csv
sys.dont_write_bytecode = True

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "data").is_dir()),
    None,
)

CSV_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "resultados"

summary_path = CSV_DIR / "resumen_simulacion.csv"

with summary_path.open("r", newline="", encoding="utf-8") as file:
    summary_rows = csv.DictReader(file)
    metrics = {row["metrica"]: row["valor"] for row in summary_rows}

L = metrics["l"]

initial_path = CSV_DIR / "particulas.csv"
initial_rows = []

with initial_path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    required_columns = {"x", "y", "vx", "vy"}
    
    for row in reader:
        initial_rows.append([float(row[column]) for column in ("x", "y", "vx", "vy")])

initial = np.asarray(initial_rows, dtype=float)

n = len(initial)
positions = initial[:, :2]
velocities = initial[:, 2:4]
speed = np.hypot(velocities[:, 0], velocities[:, 1])


print(f"N={n}; epsilon={metrics['epsilon']:g}; sigma={metrics['sigma']:g}; "
      f"dt={metrics['dt']:g}; tf={metrics['tf']:g}; método={metrics['metodo']}; "
      f"l={L:g}; ancho bin g(r)={metrics['ancho_bin_gr']:g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(positions[:, 0], positions[:, 1], c=speed, s=16, cmap="viridis")
axes[0].set(title=f"Configuración inicial — epsilon={metrics['epsilon']:g}, sigma={metrics['sigma']:g}",
            xlabel="x", ylabel="y",
            xlim=(0, L), ylim=(0, L), aspect="equal")
axes[0].grid(alpha=0.2)
fig.colorbar(axes[0].collections[0], ax=axes[0], label="Rapidez")

stride = max(1, n // 500)
direction = velocities[::stride].copy()
direction /= np.maximum(np.linalg.norm(direction, axis=1, keepdims=True), 1e-12)
axes[1].scatter(positions[:, 0], positions[:, 1], s=9, color="#b8c4ce")
axes[1].quiver(positions[::stride, 0], positions[::stride, 1],
               direction[:, 0], direction[:, 1], color="#d1495b",
               angles="xy", scale_units="xy", scale=0.45)
axes[1].set(title="Dirección de las velocidades iniciales", xlabel="x", ylabel="y",
            xlim=(0, L), ylim=(0, L), aspect="equal")
axes[1].grid(alpha=0.2)
plt.tight_layout()
fig.savefig(FIG_DIR / "configuracion_y_velocidades.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].hist(speed, bins="fd", color="#3973ac", edgecolor="white")
axes[0].set(title="Distribución de rapidez", xlabel="|v|", ylabel="Partículas")
axes[1].scatter(velocities[:, 0], velocities[:, 1], s=10, alpha=0.55)
axes[1].set(title="Componentes de velocidad", xlabel="$v_x$", ylabel="$v_y")
hb = axes[2].hexbin(positions[:, 0], positions[:, 1], gridsize=35,
                    extent=(0, L, 0, L), mincnt=1, cmap="magma")
axes[2].set(title=f"Densidad espacial inicial (l={L:g})", xlabel="x", ylabel="y",
            xlim=(0, L), ylim=(0, L), aspect="equal")
fig.colorbar(hb, ax=axes[2], label="Partículas por celda")
for axis in axes:
    axis.grid(alpha=0.15)
plt.tight_layout()
fig.savefig(FIG_DIR / "distribuciones_iniciales.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
energy_names = ["Hamiltoniano inicial", "Hamiltoniano final"]
energy_values = [metrics["hamiltoniano_inicial"], metrics["hamiltoniano_final"]]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(energy_names, energy_values, color=["#3973ac", "#d1495b"])
ax.set_ylabel("Energía")
ax.set_title(f"Hamiltonianos calculados en C++ ({metrics['metodo']})")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
fig.savefig(FIG_DIR / "hamiltoniano_inicial_final.png", dpi=160, bbox_inches="tight")
plt.show()
print(f"Hamiltoniano inicial: {energy_values[0]:.8g}")
print(f"Hamiltoniano final: {energy_values[1]:.8g}")

In [ ]:
cost_names = ["Ubicación", "Fuerzas", "Componentes de fuerza x/y", "Hamiltoniano",
              "Distribución radial", "Total de visitas a pares", "Total contabilizado"]
cost_keys = ["comparaciones_ubicacion", "evaluaciones_pares_fuerza",
             "evaluaciones_componentes_fuerza", "evaluaciones_pares_hamiltoniano",
             "evaluaciones_pares_distribucion", "total_visitas_pares",
             "trabajo_incluyendo_componentes"]

cost_values = [metrics[key] for key in cost_keys]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cost_names, cost_values, color=["#7a9e9f", "#d1495b", "#edae49",
                                                 "#3973ac", "#86bbd8", "#33658a", "#2f4858"])
ax.set_xscale("log")
ax.set_xlabel("Evaluaciones de pares (escala logarítmica)")
ax.set_title(f"Coste calculado en C++ — {metrics['metodo']}, N={n}, "
             f"S={int(metrics['configuraciones'])}, dt={metrics['dt']:g}")
ax.grid(axis="x", which="both", alpha=0.2)
ax.bar_label(bars, labels=[f"{value:,.0f}" for value in cost_values], padding=4, fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "coste_computacional.png", dpi=160, bbox_inches="tight")
plt.show()
print("Trabajo contabilizado por C++:", f"{metrics['trabajo_incluyendo_componentes']:,.0f}")